In [1]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# lerobot
from lerobot.record import RecordConfig, DatasetRecordConfig
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.constants import OBS_ENV_STATE 

# torch
from torch import cuda

# utils
import pprint
from dotenv import load_dotenv

# my code
from robot.robot_config import robot_config
from robot.robot_const import FOLDED_START_POSE
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR, DATASETS_DIR
from src.record_extended import record_extended
from src.utils import check_resume

# yolo
from src.yolo.yolo_lerobot_processor import YoloAnnotateProcessorStep

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


#### 1. Set Evaluation Params:

In [2]:
# global settings
PUSH_TO_HUB              = False
OVERRIDE_POLICY_FEATURES = False      # used when I need to remove visual features from the policy
SAVE_TO_DATASET          = True
NUM_EPISODES             = 5
FPS                      = 30
EPISODE_TIME_SEC         = 60
RESET_TIME_SEC           = 5
EVAL_TYPE                = 'real_v0'

# ACT policy settings
ACT_REPO_NAME     = 'so101_car_pick_and_place-bbox'
ACT_EXPERIMENT_NAME   = 'yolo_v0'
POLICY_CHECKPOINT = '060000'
POLICY_TYPE       = 'act'

# YOLO model settings
YOLO_REPO_NAME       = 'so101_car_pick_and_place'
YOLO_EXPERIMENT_NAME = 'v0'

Log to HF

In [3]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

#### 2. Initialize Policy and Overrides:

In [4]:
assert len(POLICY_CHECKPOINT) == 6

# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / ACT_REPO_NAME / ACT_EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    print('Loading policy from HF')
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{ACT_REPO_NAME}-{ACT_EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config.input_features)

{'observation.environment_state': PolicyFeature(type=<FeatureType.ENV: 'ENV'>,
                                                shape=(6,)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                             shape=(3, 480, 640)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                               shape=(3, 480, 640)),
 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                    shape=(6,))}


Policy Overrides:

In [5]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
# policy_config.n_action_steps = 100
# policy_config.temporal_ensemble_coeff = 0.01
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 100, temporal_ensemble = None


### 3. Build YOLO Model
Fetch model:

In [6]:
yolo_model_path = POLICIES_DIR / 'yolo' / YOLO_REPO_NAME / YOLO_EXPERIMENT_NAME / 'best.pt'
if not yolo_model_path.exists():
    print('Loading yolo model from HF')
    model_id = f"{HF_NAME}/yolov8s-obb-{YOLO_REPO_NAME}-{YOLO_EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = model_id,
    revision               = "main",
    local_dir              = str(yolo_model_path.parent),
    local_dir_use_symlinks = False
    )

In [7]:
yolo_processor = YoloAnnotateProcessorStep(model_path = yolo_model_path, cam_name = 'top_cam')

#### 4. Build Evaluation Dataset
Used for data storage of the evaluation dataset

In [ ]:
dataset_path = EVAL_DIR / POLICY_TYPE / ACT_REPO_NAME / f"{ACT_EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{ACT_REPO_NAME}_{ACT_EXPERIMENT_NAME}-{EVAL_TYPE}",
    single_task                         = f"eval dataset for {ACT_REPO_NAME} with policy {ACT_EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

Add new feature if not included in ds:

In [9]:
new_feature_name = OBS_ENV_STATE
new_feature_meta= {
    "dtype": "float32",
    "names": [
        "source_x",
        "source_y",
        "source_r",
        "target_x",
        "target_y",
        "target_r"
    ],
    "shape": (6,)
}

new_feature_list = [{new_feature_name: new_feature_meta}]

#### 5. Run inference while calling Gemini for environemnt evaluation

In [ ]:
rc.resume = check_resume(dataset_path)
record_extended(
    rc, 
    extra_robot_processor = [yolo_processor],
    extra_features        = new_feature_list,
    save_to_ds            = SAVE_TO_DATASET,
    reset_pose            = FOLDED_START_POSE,
    give_feedback         = False,
    log_to_file           = False,
    replace_image_key     = "observation.images.top_cam",
    replace_with_key      = "top_cam_bbox",
)

Loading weights from local directory


INFO 2025-11-26 21:29:09 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-11-26 21:29:11 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-11-26 21:29:11 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-11-26 21:29:12 ls/utils.py:227 Recording episode 6


Escape key pressed. Stopping data recording...


Map: 100%|██████████| 642/642 [00:00<00:00, 1106.42 examples/s]
Generating train split: 4629 examples [00:00, 782722.56 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-fram